In [1]:
from tqsdk.ta import MA, EMA

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

In [ ]:
import warnings
warnings.simplefilter(action='ignore',category=pd.errors.PerformanceWarning)

In [39]:
%run CommonUtil.ipynb

# 参数

In [3]:
# 需要封装 传入函数
# 对均线的短中长周期设参数
# s1,s2,s3 = 4, 8, 24
# s1,s2,s3 = 8, 16, 48
# s1,s2,s3 = 12, 24, 72
# s1,s2,s3 = 20,40,120
s0=3
# s1,s2,s3 = 8,16,32
# m1,m2,m3 = 32,64,128
# l1,l2,l3 = 128,256,512
# s1,s2,s3 = 8,13,21
# m1,m2,m3 = 21,34,55
# l1,l2,l3 = 55,89,144
# 8,16,32,64,128,256,512

s1,s2,s3 = 6,12,24
m1,m2,m3 = 24,48,96
l1,l2,l3 = 96,192,384


kls=0.2
klm=1
kls=1.2
zig1=0.2/100
zig2=0.4/100
zig3=0.6/100
zig4=0.9/100
yc_vol_ratio=2 # 异常vol ratio

# 对长上影/下影分类  ma_body 的2倍以上就是
ups,lows = 2,2
#回踩反抽
huicai_range = 0.5
# 对vol 最高第二高进行定位
vs=13  #量的变化更快，采用更短的周期
vm=46
vl=150
vThresh = 0.15

# 用于计算买卖压的
body_thresh=0.5
bsp_thresh1 = 1.4
bsp_thresh2 = 2.1
bsp_n = 20

# 计算n周期ma_body
n_ma_body=20
# 和高点比较（千分之）
peak_var_pct = 1.5 
cs=20 
cm=46
cl=112 #1 Day

# 靠拢的参数: ma_body的倍数
# ma1-3
kl_s_ratio = 1
# ma3-5
kl_m_ratio = 1.5

# read data

In [12]:
# raw = pd.read_csv('./data/ic2509_250906.csv')
raw = pd.read_csv('./data/ic2509_250912.csv')

In [13]:
klines=raw.loc[:,['time','open','high','low','close','volume']]

# calculate

In [105]:
%run CommonUtil.ipynb

In [106]:
# %time 
start = datetime.datetime.now()
base = calc_base_data(klines)
datetime.datetime.now()-start
# base=calc_base_data(klines)

ZeroDivisionError: float division by zero

In [95]:
base1=base.set_index('time')

In [96]:
base1.loc['2025/09/04-12:35':'2025/09/08-10:35',:][['ma_category_s','close','ma1','ma2','ma3','mv_stage','peak_2nd1']]
# ma_stage
# 空头 1
# 空头震荡 2
# 空转多 3
# 多头 4
# 多头震荡 5
# 多转空 6

,ma_category_s,close,ma1,ma2,ma3,mv_stage,peak_2nd1
time,,,,,,,
2025/09/04-13:05,1.0,6667.8,6686.510935,6702.545038,6729.602461,2,6777.4
2025/09/04-13:10,1.0,6703.0,6691.222096,6702.615032,6727.474264,2,6777.4
2025/09/04-13:15,1.0,6711.0,6696.872926,6703.905027,6726.156323,2,6729.8
2025/09/04-13:20,1.0,6694.2,6696.109233,6702.411946,6723.599817,2,6729.8
2025/09/04-13:25,1.0,6674.0,6689.792309,6698.040877,6719.631832,2,6729.8
2025/09/04-13:30,1.0,6671.8,6684.651649,6694.003819,6715.805285,2,6729.8
2025/09/04-13:35,1.0,6657.6,6676.922607,6688.403232,6711.148863,1,6729.8
2025/09/04-13:40,1.0,6650.4,6669.344719,6682.556581,6706.288954,1,6729.8
2025/09/04-13:45,1.0,6654.0,6664.960514,6678.163261,6702.105837,1,6729.8


# 分析

## 分析peak

In [97]:
base1.loc['2025/09/08-10:30':'2025/09/11-10:30',:][['close1', 'peak1',
       'valley1', 'p_v_value1','peak_2nd1', 'p_v_value_bw1', 'p_pos1', 'v_pos1',
       'z_pos1', 'dw_pct_last1', 'up_pct_last1', 'z_pct_prev1',
       'z_len_prev1', 'up_pct_avg1', 'dw_pct_avg1', 'z_pct_curr1',
       'z_pct_avg1', 'up_pct_avg_k1', 'dw_pct_avg_k1', 'z_len_curr1',
       'z_len_full1', 'z_pct_curr_avg_k1', 'z_pct_prev_avg_k1',
       'z_pct_avg_k1', 'v_pct_curr1', 'p_pct_curr1', 'valley_last1',
       'peak_last1', 'v_pct_bw1', 'p_pct_bw1', 'z_pct_curr_full1',
       'z_pct_curr_next1', 'zig_flag1', 'zig_index1']]

,close1,peak1,valley1,p_v_value1,peak_2nd1,p_v_value_bw1,p_pos1,v_pos1,z_pos1,dw_pct_last1,up_pct_last1,z_pct_prev1,z_len_prev1,up_pct_avg1,dw_pct_avg1,z_pct_curr1,z_pct_avg1,up_pct_avg_k1,dw_pct_avg_k1,z_len_curr1,z_len_full1,z_pct_curr_avg_k1,z_pct_prev_avg_k1,z_pct_avg_k1,v_pct_curr1,p_pct_curr1,valley_last1,peak_last1,v_pct_bw1,p_pct_bw1,z_pct_curr_full1,z_pct_curr_next1,zig_flag1,zig_index1
time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025/09/08-10:30,6878.8,6931.0,6894.6,6931.0,6927.4,6842.4,6,8,6,0.0000,5.2795,5.2795,0.0,7.509441,-6.197649,-7.531381,-6.197649,1.328321,-0.728261,6.0,11.0,-1.255230,inf,0.970686,-2.291649,-7.531381,6883.8,6927.4,-7.571143,-12.783148,-12.783148,-6.319434,0,272
2025/09/08-10:35,6887.2,6931.0,6894.6,6931.0,6927.4,6842.4,7,9,7,0.0000,5.2795,5.2795,0.0,7.509441,-6.197649,-6.319434,-6.197649,1.328321,-0.728261,7.0,11.0,-0.902776,inf,0.970686,-1.073304,-6.319434,6883.8,6927.4,-7.571143,-12.783148,-12.783148,-10.301544,0,272
2025/09/08-10:40,6859.6,6931.0,6894.6,6931.0,6927.4,6842.4,8,10,8,0.0000,5.2795,5.2795,0.0,7.509441,-6.197649,-10.301544,-6.197649,1.328321,-0.728261,8.0,11.0,-1.287693,inf,0.970686,-5.076437,-10.301544,6883.8,6927.4,-7.571143,-12.783148,-12.783148,-10.734382,0,272
2025/09/08-10:45,6856.6,6931.0,6894.6,6931.0,6927.4,6842.4,9,11,9,0.0000,5.2795,5.2795,0.0,7.509441,-6.197649,-10.734382,-6.197649,1.328321,-0.728261,9.0,11.0,-1.192709,inf,0.970686,-5.511560,-10.734382,6883.8,6927.4,-7.571143,-12.783148,-12.783148,-12.090607,0,272
2025/09/08-10:50,6847.2,6931.0,6894.6,6931.0,6927.4,6842.4,10,12,10,0.0000,5.2795,5.2795,0.0,7.509441,-6.197649,-12.090607,-6.197649,1.328321,-0.728261,10.0,11.0,-1.209061,inf,0.970686,-6.874946,-12.090607,6883.8,6927.4,-7.571143,-12.783148,-12.783148,-12.783148,0,272
2025/09/08-10:55,6842.4,6931.0,6842.4,6842.4,6927.4,6842.4,11,0,0,-12.7831,0.0000,-12.7831,11.0,7.509441,-6.197649,-12.783148,7.509441,1.328321,-0.728261,11.0,11.0,-1.162104,-1.162100,0.970686,-7.571143,-12.783148,6894.6,6927.4,0.000000,-12.783148,4.004443,4.004443,1,273
2025/09/08-11:00,6869.8,6869.8,6842.4,6869.8,6931.0,6869.8,0,1,0,0.0000,4.0044,4.0044,1.0,7.509441,-6.197649,4.004443,-6.197649,1.328321,-0.728261,1.0,1.0,4.004443,4.004400,0.970686,4.004443,-8.829895,6894.6,6931.0,4.004443,0.000000,-2.299921,-2.299921,1,274
2025/09/08-11:05,6854.0,6869.8,6854.0,6854.0,6931.0,6854.0,1,0,0,-2.2999,0.0000,-2.2999,1.0,7.509441,-6.197649,-2.299921,7.509441,1.328321,-0.728261,1.0,1.0,-2.299921,-2.299900,0.970686,NaN,-2.299921,6842.4,6931.0,0.000000,-2.299921,13.481179,0.992121,1,275
2025/09/08-11:10,6860.8,6869.8,6854.0,6854.0,6931.0,6946.4,2,1,1,-2.2999,0.0000,-2.2999,0.0,7.509441,-6.197649,0.992121,7.509441,1.328321,-0.728261,1.0,24.0,0.992121,-inf,0.970686,0.992121,-1.310082,6842.4,6931.0,13.481179,11.150252,13.481179,3.880945,0,275


In [98]:
base1.loc['2025/09/10-09:50':'2025/09/11-10:30',:][['close2', 'peak2',
       'valley2', 'p_v_value2', 'peak_2nd1','p_v_value_bw2', 'p_pos2', 'v_pos2',
       'z_pos2', 'dw_pct_last2', 'up_pct_last2', 'z_pct_prev2',
       'z_len_prev2', 'up_pct_avg2', 'dw_pct_avg2', 'z_pct_curr2',
       'z_pct_avg2', 'up_pct_avg_k2', 'dw_pct_avg_k2', 'z_len_curr2',
       'z_len_full2', 'z_pct_curr_avg_k2', 'z_pct_prev_avg_k2',
       'z_pct_avg_k2', 'v_pct_curr2', 'p_pct_curr2', 'valley_last2',
       'peak_last2', 'v_pct_bw2', 'p_pct_bw2', 'z_pct_curr_full2',
       'z_pct_curr_next2', 'zig_flag2', 'zig_index2']]

,close2,peak2,valley2,p_v_value2,peak_2nd1,p_v_value_bw2,p_pos2,v_pos2,z_pos2,dw_pct_last2,up_pct_last2,z_pct_prev2,z_len_prev2,up_pct_avg2,dw_pct_avg2,z_pct_curr2,z_pct_avg2,up_pct_avg_k2,dw_pct_avg_k2,z_len_curr2,z_len_full2,z_pct_curr_avg_k2,z_pct_prev_avg_k2,z_pct_avg_k2,v_pct_curr2,p_pct_curr2,valley_last2,peak_last2,v_pct_bw2,p_pct_bw2,z_pct_curr_full2,z_pct_curr_next2,zig_flag2,zig_index2
time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025/09/10-09:50,6934.4,6879.2,6856.2,6856.2,6866.8,6944.6,13,10,10,-3.3434,0.0000,-3.3434,0.0,10.388698,-8.397042,11.405735,10.388698,1.229735,-0.606308,10.0,11.0,1.140573,-inf,0.847184,11.405735,8.024189,6839.4,6942.2,12.893440,9.506919,12.893440,12.893440,0,183
2025/09/10-09:55,6944.6,6944.6,6856.2,6944.6,6879.2,6944.6,0,11,0,0.0000,12.8934,12.8934,11.0,10.388698,-8.397042,12.893440,-8.397042,1.229735,-0.606308,11.0,11.0,1.172131,1.172127,0.847184,12.893440,9.506919,6839.4,6879.2,12.893440,0.000000,-15.062063,-4.060709,1,184
2025/09/10-10:00,6916.4,6944.6,6856.2,6944.6,6879.2,6840.0,1,12,1,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-4.060709,-8.397042,1.229735,-0.606308,1.0,10.0,-4.060709,inf,0.847184,8.780374,-4.060709,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-7.142240,0,184
2025/09/10-10:05,6895.0,6944.6,6856.2,6944.6,6879.2,6840.0,2,13,2,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-7.142240,-8.397042,1.229735,-0.606308,2.0,10.0,-3.571120,inf,0.847184,5.659111,-7.142240,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-7.833425,0,184
2025/09/10-10:10,6890.2,6944.6,6856.2,6944.6,6879.2,6840.0,3,14,3,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-7.833425,-8.397042,1.229735,-0.606308,3.0,10.0,-2.611142,inf,0.847184,4.959015,-7.833425,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-5.759871,0,184
2025/09/10-10:15,6904.6,6944.6,6856.2,6944.6,6944.6,6840.0,4,15,4,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-5.759871,-8.397042,1.229735,-0.606308,4.0,10.0,-1.439968,inf,0.847184,7.059304,-5.759871,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-9.446188,0,184
2025/09/10-10:20,6879.0,6944.6,6856.2,6944.6,6944.6,6840.0,5,16,5,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-9.446188,-8.397042,1.229735,-0.606308,5.0,10.0,-1.889238,inf,0.847184,3.325457,-9.446188,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-11.145350,0,184
2025/09/10-10:25,6867.2,6944.6,6856.2,6944.6,6944.6,6840.0,6,17,6,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-11.145350,-8.397042,1.229735,-0.606308,6.0,10.0,-1.857558,inf,0.847184,1.604387,-11.145350,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-11.634939,0,184
2025/09/10-10:30,6863.8,6944.6,6856.2,6944.6,6944.6,6840.0,7,18,7,0.0000,12.8934,12.8934,0.0,10.388698,-8.397042,-11.634939,-8.397042,1.229735,-0.606308,7.0,10.0,-1.662134,inf,0.847184,1.108486,-11.634939,6839.4,6879.2,-2.362825,-15.062063,-15.062063,-12.671716,0,184
